# MLP and ResMLP without feature selection

In [1]:
import os
import random

import numpy as np
import torch
import torch.nn as nn
from sklearn.model_selection import train_test_split

SEED = 42
NS = (20, 30, 40)
FRACS = (0.25, 0.5, 1.0)
LRS = (1e-3, 3e-3, 1e-2)  # the same three learning rates for every architecture
BATCH_SIZE = 128
MAX_EPOCHS = 300
PATIENCE = 40              # stop after this many epochs without a better val accuracy
WEIGHT_ROOT = "./weights/mlp_no_fs"


def set_seed(seed=42):
    """Set a seed for reproducibility."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


set_seed(SEED)

## Data

In [2]:
def load(name):
    """Load (X, y) arrays for a dataset."""
    return np.load(f"Datasets/{name}-X.npy"), np.load(f"Datasets/{name}-y.npy")


def split(X, y):
    """60% train, 20% validation, 20% test."""
    X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.4, random_state=SEED)
    X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=SEED)
    return X_train, y_train, X_val, y_val, X_test, y_test


def binarise(X):
    """Map each noisy +-1 feature to its sign."""
    return np.where(X > 0, 1.0, -1.0)


def to_tensors(X, y):
    """Convert numpy arrays to float tensors."""
    return torch.from_numpy(X).float(), torch.from_numpy(y).float()

| Family | Size   | Hidden layers     |
|--------|--------|-------------------|
| MLP    | small  | (32,)             |
| MLP    | medium | (64, 64)          |
| MLP    | large  | (128, 128, 128)   |
| ResMLP | small  | (32,)             |
| ResMLP | medium | (64, 64)          |
| ResMLP | large  | (128, 128, 128)   |

In [3]:
class MLP(nn.Module):
    """Plain ReLU MLP that outputs one logit per row."""

    def __init__(self, d_in, hidden):
        super().__init__()
        layers, d = [], d_in
        for h in hidden:
            layers += [nn.Linear(d, h), nn.ReLU()]
            d = h
        self.net = nn.Sequential(*layers, nn.Linear(d, 1))

    def forward(self, x):
        return self.net(x).squeeze(1)


class ResBlock(nn.Module):
    """Pre-norm residual block: x + Linear(ReLU(Linear(LayerNorm(x))))."""

    def __init__(self, width):
        super().__init__()
        self.f = nn.Sequential(nn.LayerNorm(width), nn.Linear(width, width), nn.ReLU(), nn.Linear(width, width))

    def forward(self, x):
        return x + self.f(x)


class ResMLP(nn.Module):
    """Linear stem, `blocks` residual blocks of size `width`, then a linear head."""

    def __init__(self, d_in, width, blocks):
        super().__init__()
        self.stem = nn.Linear(d_in, width)
        self.blocks = nn.Sequential(*[ResBlock(width) for _ in range(blocks)])
        self.head = nn.Sequential(nn.LayerNorm(width), nn.ReLU(), nn.Linear(width, 1))

    def forward(self, x):
        return self.head(self.blocks(self.stem(x))).squeeze(1)


# family -> list of (architecture name, kwargs for the constructor)
FAMILIES = {
    "mlp": (MLP, [("small", dict(hidden=(32,))),
                  ("medium", dict(hidden=(64, 64))),
                  ("large", dict(hidden=(128, 128, 128)))]),
    "resmlp": (ResMLP, [("small", dict(width=32, blocks=1)),
                        ("medium", dict(width=64, blocks=2)),
                        ("large", dict(width=128, blocks=3))]),
}

for family, (cls, archs) in FAMILIES.items():
    for arch, kw in archs:
        n_params = [sum(p.numel() for p in cls(n, **kw).parameters()) for n in NS]
        print(f"{family:<7} {arch:<7} {kw}  parameters for {NS} inputs: {n_params}")


def accuracy(net, X, y):
    """Accuracy of `net` on tensors `X`, `y`."""
    net.eval()
    with torch.no_grad():
        return float(((net(X) > 0).float() == y).float().mean())


def train(net, X_tr, y_tr, X_va, y_va, lr):
    """Adam + BCE with early stopping on val accuracy; returns (best val acc, best epoch)."""
    set_seed(SEED)
    opt = torch.optim.Adam(net.parameters(), lr=lr)
    loss_fn = nn.BCEWithLogitsLoss()
    gen = torch.Generator().manual_seed(SEED)
    best_acc, best_ep, best_state = -1.0, 0, None
    for ep in range(1, MAX_EPOCHS + 1):
        net.train()
        for idx in torch.randperm(len(y_tr), generator=gen).split(BATCH_SIZE):
            loss = loss_fn(net(X_tr[idx]), y_tr[idx])
            opt.zero_grad()
            loss.backward()
            opt.step()
        acc = accuracy(net, X_va, y_va)
        if acc > best_acc:
            best_acc, best_ep = acc, ep
            best_state = {k: v.detach().clone() for k, v in net.state_dict().items()}
        elif ep - best_ep >= PATIENCE:
            break
    net.load_state_dict(best_state)
    return best_acc, best_ep


def tune(family, train_data, val_data):
    """Train every architecture with every lr in LRS; keep the best on val (first wins ties)."""
    cls, archs = FAMILIES[family]
    d_in = train_data[0].shape[1]
    best = None
    for arch, kw in archs:
        for lr in LRS:
            set_seed(SEED)
            net = cls(d_in, **kw)
            val_acc, ep = train(net, *train_data, *val_data, lr)
            print(f"    {family:<7} {arch:<7} lr {lr:<6g} val {val_acc:.4f}  (best epoch {ep})")
            if best is None or val_acc > best[0]:
                best = (val_acc, dict(arch=arch, **kw, lr=lr, epoch=ep), net)
    return best

mlp     small   {'hidden': (32,)}  parameters for (20, 30, 40) inputs: [705, 1025, 1345]
mlp     medium  {'hidden': (64, 64)}  parameters for (20, 30, 40) inputs: [5569, 6209, 6849]
mlp     large   {'hidden': (128, 128, 128)}  parameters for (20, 30, 40) inputs: [35841, 37121, 38401]
resmlp  small   {'width': 32, 'blocks': 1}  parameters for (20, 30, 40) inputs: [2945, 3265, 3585]
resmlp  medium  {'width': 64, 'blocks': 2}  parameters for (20, 30, 40) inputs: [18433, 19073, 19713]
resmlp  large   {'width': 128, 'blocks': 3}  parameters for (20, 30, 40) inputs: [102913, 104193, 105473]


## All datasets and scales

In [4]:
def run(n, frac):
    """Use the first `frac` of train/val with all n features, tune both families, save each best model."""
    X_train, y_train, X_val, y_val, X_test, y_test = split(*load(f"kryptonite-{n}"))
    m, mv = int(frac * len(y_train)), int(frac * len(y_val))
    S_tr, y_tr = binarise(X_train[:m]), y_train[:m]
    S_va, y_va = binarise(X_val[:mv]), y_val[:mv]
    S_te, y_te = binarise(X_test), y_test
    print(f"\n=== n={n}, {frac:.0%}: {m} train / {mv} val / {len(y_te)} test ===")

    cols = list(range(n))  # no feature selection: all features are used
    tr = to_tensors(S_tr, y_tr)
    va = to_tensors(S_va, y_va)
    te = to_tensors(S_te, y_te)

    rows = []
    for family in FAMILIES:
        val_acc, params, net = tune(family, tr, va)
        test_acc = accuracy(net, *te)
        out_dir = os.path.join(WEIGHT_ROOT, family, f"n{n}", f"{round(frac * 100)}pct")
        os.makedirs(out_dir, exist_ok=True)
        path = os.path.join(out_dir, "model.pt")
        torch.save({"state_dict": net.state_dict(), "features": cols, "params": params}, path)
        print(f"  best {family:<7} {params}  val {val_acc:.4f}  test {test_acc:.4f}  -> {path}")
        rows.append({"n": n, "frac": frac, "model": family, "train_size": m, "features": cols,
                     "params": params, "val": val_acc, "test": test_acc, "path": path})
    return rows


results = [row for n in NS for frac in FRACS for row in run(n, frac)]


=== n=20, 25%: 1800 train / 600 val / 2400 test ===
    mlp     small   lr 0.001  val 0.5483  (best epoch 28)
    mlp     small   lr 0.003  val 0.5500  (best epoch 2)
    mlp     small   lr 0.01   val 0.5817  (best epoch 110)
    mlp     medium  lr 0.001  val 0.5617  (best epoch 36)
    mlp     medium  lr 0.003  val 0.5617  (best epoch 22)
    mlp     medium  lr 0.01   val 0.5917  (best epoch 28)
    mlp     large   lr 0.001  val 0.5533  (best epoch 6)
    mlp     large   lr 0.003  val 0.5517  (best epoch 16)
    mlp     large   lr 0.01   val 0.5867  (best epoch 47)
  best mlp     {'arch': 'medium', 'hidden': (64, 64), 'lr': 0.01, 'epoch': 28}  val 0.5917  test 0.5987  -> ./weights/mlp_no_fs\mlp\n20\25pct\model.pt
    resmlp  small   lr 0.001  val 0.5750  (best epoch 29)
    resmlp  small   lr 0.003  val 0.6433  (best epoch 67)
    resmlp  small   lr 0.01   val 0.7633  (best epoch 98)
    resmlp  medium  lr 0.001  val 0.5600  (best epoch 21)
    resmlp  medium  lr 0.003  val 0.6133  (

## Summary

In [5]:
print(f"{'n':>3} {'frac':>5} {'model':<7} {'arch':<7} {'lr':>6} {'val':>7} {'test':>7}")
for r in results:
    p = r["params"]
    print(f"{r['n']:>3} {r['frac']:>5.0%} {r['model']:<7} {p['arch']:<7} {p['lr']:>6g} "
          f"{r['val']:>7.4f} {r['test']:>7.4f}")

  n  frac model   arch        lr     val    test
 20   25% mlp     medium    0.01  0.5917  0.5987
 20   25% resmlp  medium    0.01  0.8550  0.8504
 20   50% mlp     large     0.01  0.9158  0.9171
 20   50% resmlp  large     0.01  0.9508  0.9454
 20  100% mlp     large    0.003  0.9354  0.9367
 20  100% resmlp  medium    0.01  0.9533  0.9554
 30   25% mlp     large     0.01  0.6140  0.5770
 30   25% resmlp  large     0.01  0.9190  0.9087
 30   50% mlp     large     0.01  0.9270  0.9365
 30   50% resmlp  medium   0.003  0.9430  0.9352
 30  100% mlp     large    0.003  0.9435  0.9452
 30  100% resmlp  large    0.003  0.9550  0.9557
 40   25% mlp     small     0.01  0.5794  0.5398
 40   25% resmlp  large     0.01  0.9056  0.9019
 40   50% mlp     large     0.01  0.9137  0.9111
 40   50% resmlp  large    0.003  0.9394  0.9423
 40  100% mlp     medium   0.003  0.9458  0.9527
 40  100% resmlp  large    0.003  0.9522  0.9569


## Loading a saved model

In [6]:
def load_model(family, n, frac):
    """Rebuild a saved best model and return (net, feature indices)."""
    ckpt = torch.load(os.path.join(WEIGHT_ROOT, family, f"n{n}", f"{round(frac * 100)}pct", "model.pt"))
    cls, archs = FAMILIES[family]
    net = cls(len(ckpt["features"]), **dict(archs)[ckpt["params"]["arch"]])
    net.load_state_dict(ckpt["state_dict"])
    return net.eval(), ckpt["features"]


# sanity check: the reloaded models reproduce the reported test accuracy
for r in results:
    net, cols = load_model(r["model"], r["n"], r["frac"])
    _, _, _, _, X_test, y_test = split(*load(f"kryptonite-{r['n']}"))
    assert abs(accuracy(net, *to_tensors(binarise(X_test)[:, cols], y_test)) - r["test"]) < 1e-9
print("all saved models reload and reproduce their test accuracy")

all saved models reload and reproduce their test accuracy
